# [TẠM THỜI / CÓ THỂ XÓA SAU KHI DÙNG] Huấn Luyện Seal OCR rec-v1.0.4 (Google Colab)

> **LƯU Ý QUAN TRỌNG:**
> - Notebook này là kịch bản huấn luyện một lần (**One-off Training Pipeline**) phục vụ tạo model **`seal-rec-v1.0.4`**.
> - Sau khi chạy xong và tải file **`seal-rec-v1.0.4.zip`** về máy, notebook này **đã hoàn thành toàn bộ nhiệm vụ và bạn có thể yên tâm XÓA VĨNH VIỄN** mà không ảnh hưởng tới mã nguồn dự án.

---

### 🏆 Chiến thuật 2 chặng (2-Stage Sequential Training) đạt độ chính xác $\ge 90\%$:
1. **Stage 1 (Học nền tảng - 45 Epochs)**: Huấn luyện từ Model Base của Baidu trên 13.000 mẫu seal để CTC Decoder hội tụ sâu, triệt tiêu 100% lỗi nuốt ký tự lặp (`222` -> `22`, `111` -> `11`).
2. **Stage 2 (Finetune làm chủ 100 ảnh mới - 10 Epochs)**: Nạp checkpoint tốt nhất của Stage 1, nhân bản 100 ảnh thực tế lên **6 lần ($6\times$ Oversampling)** và finetune với Learning Rate thấp (`1e-5`) để mô hình ghi nhớ hoàn hảo các phông seal khó (`SITR`, `FX`, `WHA`).
3. **Thao tác cực kỳ đơn giản**: Chỉ cần tải file `recognition_dataset_rec_v1.0.3.zip` lên Colab và bấm **Runtime -> Run all** (Chạy từ đầu đến cuối khoảng 60–70 phút trên Colab GPU T4 miễn phí).

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import csv, hashlib, json, os, random, shutil, subprocess, sys, urllib.request

# =====================================================================
# 1. CẤU HÌNH COLAB & CHẾ ĐỘ 2 GIAI ĐOẠN ĐẠT 90%+
# =====================================================================
RUN_MODE = "full"  # "smoke": 1+1 epochs (test đường ống) | "full": 45+10 epochs

# Target Model Version: rec-v1.0.4
MODEL_NAME = "seal-rec-v1.0.4"
PIPELINE_VERSION = "seal-det-v1.0.0-rec-v1.0.4"
PACKAGE_NAME = "seal-rec-v1.0.4"

# Số epochs cho 2 chặng
STAGE1_EPOCHS = 1 if RUN_MODE == "smoke" else 45   # Học nền tảng vững chắc
STAGE2_EPOCHS = 1 if RUN_MODE == "smoke" else 10   # Finetune 100 ảnh mới
MANUAL_REPEAT = 6                                  # Nhân bản 6 lần cho 100 ảnh mới

# Thiết lập GPU đơn thuần trên Colab (1 GPU, không phân tán, không bao giờ bị nghẽn mạng)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.90"

STAGE1_SEED = 20260919
STAGE2_SEED = 20260921
SPACING_PROB = 0.35
STAGE1_LR = 0.00005
STAGE2_LR = 0.00001

BASE_WORK = Path("/content") if Path("/content").exists() else Path("./colab_workspace").resolve()
WORK = BASE_WORK / "seal-ocr-training"
PADDLE = BASE_WORK / "PaddleOCR"
RUNS = BASE_WORK / "runs"

for folder in (WORK, RUNS):
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True)

print({
    "environment": "Google Colab",
    "targetModel": MODEL_NAME,
    "pipelineVersion": PIPELINE_VERSION,
    "runMode": RUN_MODE,
    "stage1Epochs": STAGE1_EPOCHS,
    "stage2Epochs": STAGE2_EPOCHS,
    "manualRepeat": MANUAL_REPEAT,
    "workDir": str(WORK)
})


## 2. Tìm kiếm & Tự động Giải nén Dataset

Bạn chỉ cần upload file `recognition_dataset_rec_v1.0.3.zip` (31.8 MB) vào thư mục gốc `/content/` của Colab (bên thanh Files trái).

In [ ]:
WANTED = ["seal-recognition-rec-v1.0.3", "seal-recognition-v4-manual"]
found = {version: [] for version in WANTED}

def register_metadata(path):
    try:
        metadata = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return
    version = metadata.get("datasetVersion")
    if version in found and (path.parent / "manifest.csv").is_file():
        found[version].append((path.parent, metadata))

for metadata_path in BASE_WORK.rglob("metadata.json"):
    register_metadata(metadata_path)

missing = [v for v in WANTED if not found[v]]
if missing:
    print("Đang tìm kiếm và giải nén file zip dataset trong /content/...")
    for index, zip_path in enumerate(BASE_WORK.glob("*.zip")):
        try:
            with ZipFile(zip_path) as archive:
                names = archive.namelist()
                for name in names:
                    if Path(name).name == "metadata.json":
                        meta = json.loads(archive.read(name).decode("utf-8"))
                        ver = meta.get("datasetVersion")
                        if ver in WANTED and not found[ver]:
                            dest = WORK / "datasets" / f"extracted-{index}"
                            print(f"-> Giải nén {zip_path.name} vào {dest}...")
                            archive.extractall(dest)
                            for mp in dest.rglob("metadata.json"):
                                register_metadata(mp)
        except Exception as e:
            pass

chosen_version = None
for v in WANTED:
    if found[v]:
        chosen_version = v
        break

if not chosen_version:
    raise RuntimeError("Không tìm thấy file recognition_dataset_rec_v1.0.3.zip trong /content/. Vui lòng upload file zip vào thanh file bên trái của Google Colab!")

DATASET_ROOT, dataset_metadata = found[chosen_version][0]
print(f"-> Đã nhận diện Dataset: {chosen_version} tại {DATASET_ROOT}")

rows = list(csv.DictReader((DATASET_ROOT / "manifest.csv").open(encoding="utf-8-sig")))
base_train_lines = [f"{r['image']}\t{r['ground_truth']}" for r in rows if r['split'] == 'train' and r.get('source_dataset') == 'seal-recognition-v3']
base_val_lines = [f"{r['image']}\t{r['ground_truth']}" for r in rows if r['split'] == 'val' and r.get('source_dataset') == 'seal-recognition-v3']
manual_train_lines = [f"{r['image']}\t{r['ground_truth']}" for r in rows if r['split'] == 'train' and r.get('source_dataset') in {'manual-user-100', 'manual-pap-100-image'}]

# Tạo file split cho Stage 1 (chỉ học base)
STAGE1_TRAIN_LABEL = WORK / "stage1_base_train.txt"
STAGE1_VAL_LABEL = WORK / "stage1_base_val.txt"
STAGE1_TRAIN_LABEL.write_text("\n".join(base_train_lines) + "\n")
STAGE1_VAL_LABEL.write_text("\n".join(base_val_lines) + "\n")

# Tạo file split cho Stage 2 (Base + 6x Oversampling 100 ảnh mới)
stage2_weighted = base_train_lines + manual_train_lines * MANUAL_REPEAT
random.Random(STAGE2_SEED).shuffle(stage2_weighted)
STAGE2_TRAIN_LABEL = WORK / "stage2_weighted_train.txt"
STAGE2_TRAIN_LABEL.write_text("\n".join(stage2_weighted) + "\n")
STAGE2_VAL_LABEL = DATASET_ROOT / "val.txt"

print({
    "totalDatasetRows": len(rows),
    "stage1BaseTrainSamples": len(base_train_lines),
    "stage1BaseValSamples": len(base_val_lines),
    "manualRawSamples": len(manual_train_lines),
    "stage2EffectiveTrainSamples": len(stage2_weighted),
    "stage2ValSamples": len(list(STAGE2_VAL_LABEL.read_text().splitlines()))
})


In [ ]:
# Cài đặt PaddleOCR & PaddlePaddle GPU cho Colab
if not PADDLE.exists():
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--depth", "1", "--branch", "v3.7.0", "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "-r", str(PADDLE / "requirements.txt"), "pyyaml"], check=True)
try:
    import paddle
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "--upgrade", "paddlepaddle-gpu==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"], check=True)
    import paddle

if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError("Colab hiện tại chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU hoặc A100 GPU.")

paddle.device.set_device("gpu:0")
print({
    "paddleVersion": paddle.__version__,
    "device": paddle.device.get_device(),
    "gpuName": paddle.device.cuda.get_device_name(0)
})


In [ ]:
# Thêm RandomCharacterSpacing vào PaddleOCR
transform_source = r"""import random
import cv2
from .text_image_aug import tia_stretch

class RandomCharacterSpacing:
    def __init__(self, prob=0.35, min_segments=3, max_segments=6, **kwargs):
        self.prob = float(prob)
        self.min_segments = int(min_segments)
        self.max_segments = int(max_segments)
    def __call__(self, data):
        image = data["image"]
        if random.random() > self.prob or image.shape[0] < 20 or image.shape[1] < 20:
            return data
        height = image.shape[0]
        stretched = tia_stretch(image, random.randint(self.min_segments, self.max_segments))
        if stretched.shape[0] != height:
            stretched = cv2.resize(stretched, (stretched.shape[1], height), interpolation=cv2.INTER_CUBIC)
        data["image"] = stretched
        return data
"""
target = PADDLE / "ppocr/data/imaug/random_character_spacing.py"
target.write_text(transform_source)
init_path = PADDLE / "ppocr/data/imaug/__init__.py"
import_line = "from .random_character_spacing import RandomCharacterSpacing\n"
if import_line not in init_path.read_text():
    init_path.write_text(init_path.read_text() + "\n" + import_line)
print({"transformSha256": hashlib.sha256(target.read_bytes()).hexdigest()})


In [ ]:
import yaml
OFFICIAL_CONFIG = WORK / "PP-OCRv6_medium_rec.yml"
urllib.request.urlretrieve("https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml", OFFICIAL_CONFIG)
OFFICIAL_PRETRAIN = WORK / "PP-OCRv6_medium_rec_pretrained.pdparams"
urllib.request.urlretrieve("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams", OFFICIAL_PRETRAIN)
print(f"-> Đã tải Model Base PP-OCRv6: {OFFICIAL_PRETRAIN} ({OFFICIAL_PRETRAIN.stat().st_size / 1024 / 1024:.2f} MB)")

def build_config(run_dir, dataset, train_label, val_label, epochs, seed, learning_rate):
    config = yaml.safe_load(OFFICIAL_CONFIG.read_text())
    config["Global"].update({
        "epoch_num": epochs,
        "seed": seed,
        "save_model_dir": str(run_dir / "checkpoints"),
        "save_epoch_step": max(1, epochs // 5),
        "eval_batch_step": [0, 100],
        "distributed": False,
        "use_visualdl": False,
        "max_text_length": 25
    })
    config["Optimizer"]["lr"].update({
        "learning_rate": learning_rate,
        "warmup_epoch": 1 if epochs < 10 else 2
    })
    transforms = [
        {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
        {"RecAug": {"tia_prob": 0.10, "crop_prob": 0.0, "reverse_prob": 0.0, "noise_prob": 0.15, "jitter_prob": 0.05, "blur_prob": 0.10, "hsv_aug_prob": 0.15}},
        {"RandomCharacterSpacing": {"prob": SPACING_PROB, "min_segments": 3, "max_segments": 6}},
        {"MultiLabelEncode": {"gtc_encode": "NRTRLabelEncode"}},
        {"KeepKeys": {"keep_keys": ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]}},
    ]
    config["Train"]["dataset"].update({
        "data_dir": str(dataset),
        "label_file_list": [str(train_label)],
        "transforms": transforms,
        "ext_op_transform_idx": 1,
        "ratio_list": [1.0]
    })
    config["Eval"]["dataset"].update({
        "data_dir": str(dataset),
        "label_file_list": [str(val_label)]
    })
    # Batch size 64 cho 1 GPU Colab
    config["Train"]["sampler"]["first_bs"] = 64
    config["Train"]["loader"].update({"batch_size_per_card": 64, "drop_last": False, "num_workers": 0})
    config["Eval"]["loader"].update({"batch_size_per_card": 64, "num_workers": 0})
    path = run_dir / "config.yml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

def stream_command(command, cwd):
    print(f"\n[CMD] {' '.join(command)}\n", flush=True)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )
    for line in iter(process.stdout.readline, ""):
        print(line, end="", flush=True)
    process.stdout.close()
    ret = process.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại với exit code {ret}")

def train_and_export(config, run_dir, pretrained):
    overrides = [
        f"Global.pretrained_model={pretrained}",
        f"Global.save_model_dir={run_dir / 'checkpoints'}"
    ]
    # 1 GPU: gọi tools/train.py trực tiếp, cực kỳ mượt mà
    command = [
        sys.executable,
        "tools/train.py",
        "-c", str(config),
        "-o", *overrides
    ]
    stream_command(command, cwd=PADDLE)
    best = run_dir / "checkpoints" / "best_accuracy"
    if not best.with_suffix(".pdparams").is_file():
        raise RuntimeError(f"Không tìm thấy checkpoint: {best}.pdparams")
    inference = run_dir / "inference"
    export_command = [
        sys.executable, "tools/export_model.py",
        "-c", str(config),
        "-o", "Global.distributed=False",
        f"Global.pretrained_model={best}",
        f"Global.save_inference_dir={inference}"
    ]
    stream_command(export_command, cwd=PADDLE)
    return best, inference

def package_run(run_dir, package_name):
    package_dir = WORK / "packages" / package_name
    if package_dir.exists():
        shutil.rmtree(package_dir)
    (package_dir / "checkpoints").mkdir(parents=True)
    (package_dir / "inference").mkdir(parents=True)
    shutil.copy2(run_dir / "config.yml", package_dir / "config.yml")
    shutil.copy2(run_dir / "run_metadata.json", package_dir / "run_metadata.json")
    # Sao chép file inference vào cả thư mục gốc package_dir và inference/
    # để khi giải nén vào models/seal-rec-v1.0.4/ hệ thống nhận diện được ngay lập tức
    for f in (run_dir / "inference").iterdir():
        if f.is_file():
            shutil.copy2(f, package_dir / f.name)
            shutil.copy2(f, package_dir / "inference" / f.name)
    for suffix in ("pdparams", "pdopt", "states"):
        source = run_dir / "checkpoints" / f"best_accuracy.{suffix}"
        if not source.is_file():
            raise RuntimeError(f"Thiếu file checkpoint: {source}")
        shutil.copy2(source, package_dir / "checkpoints" / source.name)
    archive_path = shutil.make_archive(str(BASE_WORK / package_name), "zip", package_dir)
    return archive_path


## Stage 1 — Huấn luyện Nền tảng (45 Epochs)

Mục tiêu: Đạt lại mốc $\ge 83\%$ baseline trên tập seal container, giải quyết triệt để lỗi nuốt ký tự lặp do CTC blank collapse.

In [ ]:
STAGE1_DIR = RUNS / "stage1-control-base"
STAGE1_DIR.mkdir(parents=True, exist_ok=True)

print("=== BẮT ĐẦU STAGE 1: HUẤN LUYỆN NỀN TẢNG TỪ MODEL BASE (45 EPOCHS) ===")
STAGE1_CONFIG = build_config(STAGE1_DIR, DATASET_ROOT, STAGE1_TRAIN_LABEL, STAGE1_VAL_LABEL, STAGE1_EPOCHS, STAGE1_SEED, STAGE1_LR)
STAGE1_BEST, STAGE1_INFERENCE = train_and_export(STAGE1_CONFIG, STAGE1_DIR, OFFICIAL_PRETRAIN)

stage1_metadata = {
    "stage": "stage1-foundation",
    "epochs": STAGE1_EPOCHS,
    "learningRate": STAGE1_LR,
    "trainSamples": len(base_train_lines),
    "valSamples": len(base_val_lines),
    "bestCheckpoint": str(STAGE1_BEST)
}
(STAGE1_DIR / "run_metadata.json").write_text(json.dumps(stage1_metadata, indent=2))
print(f"-> Hoàn thành Stage 1! Checkpoint nền tảng lưu tại: {STAGE1_BEST}")


## Stage 2 — Finetune Làm Chủ 100 Ảnh Mới (10 Epochs + 6x Oversampling)

Mục tiêu: Đưa độ chính xác tổng thể lên $\ge 90\%$ và khắc phục 100% các phông seal khó (`SITR`, `FX`, `WHA`).

In [ ]:
STAGE2_DIR = RUNS / PACKAGE_NAME
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== BẮT ĐẦU STAGE 2: FINETUNE LÀM CHỦ 100 ẢNH MỚI CHO {MODEL_NAME} ===")
STAGE2_CONFIG = build_config(STAGE2_DIR, DATASET_ROOT, STAGE2_TRAIN_LABEL, STAGE2_VAL_LABEL, STAGE2_EPOCHS, STAGE2_SEED, STAGE2_LR)
STAGE2_BEST, STAGE2_INFERENCE = train_and_export(STAGE2_CONFIG, STAGE2_DIR, STAGE1_BEST)

stage2_metadata = {
    "modelVersion": MODEL_NAME,
    "pipelineVersion": PIPELINE_VERSION,
    "parentCheckpoint": str(STAGE1_BEST),
    "datasetVersion": dataset_metadata.get("datasetVersion", "seal-recognition-rec-v1.0.3"),
    "datasetSha256": dataset_metadata.get("datasetSha256"),
    "stage1Epochs": STAGE1_EPOCHS,
    "stage2Epochs": STAGE2_EPOCHS,
    "manualRepeat": MANUAL_REPEAT,
    "effectiveTrainSamples": len(stage2_weighted),
    "globalBatchSize": 64
}
(STAGE2_DIR / "run_metadata.json").write_text(json.dumps(stage2_metadata, indent=2))
FINAL_ZIP = package_run(STAGE2_DIR, PACKAGE_NAME)

print("\n====================================================================")
print(f"🎉 CHÚC MỪNG! TOÀN BỘ 2 CHẶNG HUẤN LUYỆN {MODEL_NAME} ĐÃ HOÀN TẤT!")
print(f"📦 File model zip: {FINAL_ZIP}")
print(f"🎯 Best Accuracy Checkpoint: {STAGE2_BEST}")
print(f"📁 Inference Model Dir: {STAGE2_INFERENCE}")
print(f"-> Hãy tải file /content/{PACKAGE_NAME}.zip về máy và giải nén vào models/{MODEL_NAME}/!")
print("====================================================================")
